In [3]:
import os
import time
import threading

from spc import SPC3

In [4]:
#initialize connection to camera and set camera mode
spc3 = SPC3(SPC3.CameraMode.NORMAL)
#spc3 = SPC3(SPC3.CameraMode.ADVANCED)


In [5]:

# Acquisition parameters - half array, all else default
spc3.SetCameraPar(
    Exposure=1040,                       # ignored in Normal mode
    NFrames=1,
    NIntegFrames=1,
    NCounters=1,
    Force8bit=SPC3.State.DISABLED,
    Half_array=SPC3.State.ENABLED,    # 32x32 half array
    Signed_data=SPC3.State.DISABLED,
)

# Gate mode: COARSE on counter 1, window = 0 to 50 clock cycles (x10ns each)
spc3.SetGateMode(1, SPC3.GateMode.COARSE)
spc3.SetCoarseGateValues(Counter=1, Start=0, Stop=50)

# Sync-in trigger: enabled, 1 frame per trigger pulse
spc3.SetSyncInState(s=SPC3.State.ENABLED, frames=1)

# Apply all settings to camera
spc3.ApplySettings()


In [6]:
data_dir = r'C:\Users\SPUD1\Documents\experiment_workspace\SPAD data'
fname = 'contacq_ODMR'

In [7]:

# --- parameters ---
filename = os.path.join(data_dir, fname)
poll_interval_s  = 0.001   # 1 ms nominal poll interval (matches qudi logic timer)
error_window_s   = 120.0   # abort only if transient errors persist uninterrupted for this long
error_retry_s    = 0.001   # 1 ms sleep between retries — matches qudi timer, prevents USB flooding

# Errors from GetMemory that are transient and should be retried.
# UNABLE_CREATE_FILE can follow a COMMUNICATION_ERROR when the SDK file handle
# is briefly in a bad state — treat identically.
RETRYABLE_ERRORS = {'COMMUNICATION_ERROR', 'UNABLE_CREATE_FILE'}

stop_event = threading.Event()

def _acq_loop():
    from datetime import datetime
    from spc import SPC3Error
    t_start = time.perf_counter()
    total_bytes = 0
    error_since = None       # perf_counter of first error in current unbroken run
    total_transient_errors = 0
    last_error_print = 0.0   # rate-limit error prints to 1/s
    try:
        spc3.ContAcqToFileStart(filename)
        print(f'Acquisition started  {datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]}  ->  {filename}')
        while not stop_event.is_set():
            try:
                total_bytes += spc3.ContAcqToFileGetMemory()
                # success — reset error tracking
                if error_since is not None:
                    elapsed_err = time.perf_counter() - error_since
                    print(f'  [INFO] Recovered after {elapsed_err:.1f} s '
                          f'({total_transient_errors} errors so far)')
                    error_since = None
            except SPC3Error as e:
                err_name = str(e)
                if any(r in err_name for r in RETRYABLE_ERRORS):
                    now = time.perf_counter()
                    if error_since is None:
                        error_since = now
                    total_transient_errors += 1
                    elapsed_err = now - error_since
                    # Print at most once per second to avoid flooding the kernel
                    if now - last_error_print >= 1.0:
                        print(f'  [WARN] {err_name} ({elapsed_err:.0f} s / {error_window_s:.0f} s window, '
                              f'#{total_transient_errors} total)')
                        last_error_print = now
                    if elapsed_err >= error_window_s:
                        print(f'  [ERROR] Errors persisted {elapsed_err:.0f} s — stopping.')
                        break
                    # Sleep 1 ms — same as qudi timer interval.
                    # A failed transfer leaves data in the camera buffer, so sleeping here
                    # does NOT advance MEMORY_FULL. Spinning without sleep stresses USB
                    # and prevents recovery.
                    time.sleep(error_retry_s)
                    continue
                else:
                    raise   # re-raise unexpected errors immediately
            time.sleep(poll_interval_s)
    finally:
        # Guard stop separately — it can raise if the SDK state is broken,
        # and an exception here would mask the real error from the loop above.
        try:
            spc3.ContAcqToFileStop()
        except Exception as e_stop:
            print(f'  [WARN] ContAcqToFileStop raised: {e_stop}')
        elapsed = time.perf_counter() - t_start
        print(f'Acquisition stopped  {datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]}  |  '
              f'{elapsed:.2f} s, {total_bytes/1e6:.2f} MB written'
              + (f', {total_transient_errors} transient errors' if total_transient_errors else ''))

stop_event.clear()
acq_thread = threading.Thread(target=_acq_loop, daemon=True)
acq_thread.start()


Acquisition started  2026-03-10 18:35:47.169  ->  C:\Users\SPUD1\Documents\experiment_workspace\SPAD data\contacq_ODMR
  [WARN] COMMUNICATION_ERROR (0 s / 120 s window, #1 total)
  [WARN] COMMUNICATION_ERROR (5 s / 120 s window, #2 total)
  [WARN] COMMUNICATION_ERROR (6 s / 120 s window, #67 total)
  [WARN] COMMUNICATION_ERROR (7 s / 120 s window, #132 total)
  [WARN] COMMUNICATION_ERROR (8 s / 120 s window, #196 total)
  [WARN] COMMUNICATION_ERROR (9 s / 120 s window, #261 total)
  [WARN] COMMUNICATION_ERROR (10 s / 120 s window, #326 total)
  [WARN] COMMUNICATION_ERROR (11 s / 120 s window, #391 total)
  [WARN] COMMUNICATION_ERROR (12 s / 120 s window, #455 total)
  [WARN] UNABLE_CREATE_FILE (13 s / 120 s window, #520 total)
  [WARN] UNABLE_CREATE_FILE (14 s / 120 s window, #584 total)
  [WARN] UNABLE_CREATE_FILE (15 s / 120 s window, #649 total)
  [WARN] UNABLE_CREATE_FILE (16 s / 120 s window, #714 total)
  [WARN] UNABLE_CREATE_FILE (17 s / 120 s window, #779 total)
  [WARN] UNABLE

In [6]:
# Run this cell to stop the acquisition
stop_event.set()
acq_thread.join()
